# This notebook shows you how to generate your own DBOF


In [1]:
import datetime

# Initial set up
In the future we will want to update this to use docker or conda for end users

## These steps help you build the project on a local machine. 
If you are using nrp Jupyterhub, it is recommended you use the next block instead.

#### Build the project
- ```pip install .```

#### Install aws cli (optional)
You will want to install this if you want to manually see the data stored in the s3 bucket.

Example for installing on linux :
- ```sudo curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"```
- ```sudo unzip awscliv2.zip```
- ```sudo ./aws/install```

## Running in NRP Jupyterhub

This is for running this notebook on nrp Jupyterhub. 

This block simply builds the project and installs dependencies not already present on nrp jupyterhub.
You can safely ignore pip warnings

For serious projects, users should use conda or docker but this notebook is meant to be very simple and user friendly.

In [2]:
#Set True if running on NRP Jupyterhub
RUNNING_ON_NRP = False


if (RUNNING_ON_NRP):
    %pip install -e ../. --no-deps

    %pip install xgcm
    %pip install zarr
    %pip install boto3
    %pip install ujson
    %pip install scikit-fmm
    %pip install aiobotocore

# NOTE IF on NRP Jupyterhub, you will likely need to restart the kernel after running this block

## Set AWS credentials
If you are accessing or writing data to S3, you must set credentials.
NOTE : S3 is all that is supported currently.
This typically corresponds to an NRP S3 bucket.

windows:

- ```$env:AWS_ACCESS_KEY_ID="..."```
- ```$env:AWS_SECRET_ACCESS_KEY="..."```

unix:

- ```export AWS_ACCESS_KEY_ID=...```
- ```export AWS_SECRET_ACCESS_KEY=...```

# Dataset generation config (quick reference)

This job is fully controlled by a YAML config file. The config defines **what time range is scanned**, **how often snapshots are taken**, and **how many spatial patches are sampled per snapshot**.

### Temporal sampling
- `data.timestep_hours`
  Total time window (in hours) to scan starting from `start_record`.
  Example: `8760` = one LLC model year (336 days).

- `data.sampling_step`
  Spacing **in hours** between snapshots within the window.
  Example: `2190` with `timestep_hours=8760` → 4 evenly spaced snapshots.

- `data.start_record`
  First valid wind/forcing record (default: `1180`). You probably don't want to change this.

Each snapshot corresponds to a **single instantaneous model timestep** (not an average).

### Spatial sampling
- `sampling.sample_points_per_snapshot`
  Number of cutouts sampled per snapshot.

- `sampling.bias_to_high_gradients`
  Exponential bias favoring high-gradient regions when sampling.

### Patch geometry
- `output.target_km_res`
  Physical patch size in km. Default 150.

- `output.down_sample_res`
  Pixel resolution of the extracted cutouts.
  Must be small enough to safely resolve `target_km_res` on the LLC grid.
  Default 64

### Output / logging
- `output.bucket`, `output.folder`
  S3 location for dataset output.

- `run.run_id`
  Unique identifier for this run (used for logs and output paths).
  If the run_id you use exists on the write location (s3 bucket), your data will be appended to the existing data from previous run(s).

- `run.log_dir`
  Local directory where logs are written.

### Invariants (not configurable)
Grid topology, model cadence (144 timesteps/hour), and dataset offsets are fixed
LLC4320 constants and are enforced in code.

### Examples
See existing configs in configs/ for examples

# A note about logs
The run logs will be stored locally on your machine in the directory you specify.
- ```log_dir/run_id/```
However under the current logic, if you attempt to run the script and the specified log output path already exists, the script will fail to run. This is by design. The reasons are as follows.
- run_id is also used for the zarr dataset path. You likely do not want to send two different runs to the same dataset.
- You will likely not want to override your previous run logs.

If you want to override this logic, simply delete the existing log output path from your local machine. You can also override the run_id with a cli argument ```--run_id```

## Example run 1
- 1 year of data
- 4 time snapshots evenly spaced through the year
- 150 cutouts per timestamp

If you are running this on your laptop or pc expect it to take a while.

In [5]:
run_id = f"year_4x150_{datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')}"
print(f"Your run id is: {run_id}")
print("Make sure and use the same run_d later when accessing your data")

Your run id is: year_4x150_20260624_180005
Make sure and use the same run_d later when accessing your data


In [2]:
# note: that --run_id can overwrite the config file
# note: All Dask logs are warnings. Don't be alarmed.

!generate-llc-dataset --config ../../configs/cutouts/run/run_from_globals_example.yaml  --run_id {run_id}

JobConfig(run=RunConfig(run_id='year_4x150', log_dir='../test_logs/'), input=InputConfig(folder='surface_fields/global_SURF_test01', s3_endpoint='https://s3-west.nrp-nautilus.io', bucket='dbof', date_prefixes=['20121109_120000'], grid_access=GridAccessConfig(s3_endpoint='https://s3-west.nrp-nautilus.io', bucket='dbof', folder='native_grid_dbof_training_data', dataset_name='llc4320_grid.zarr')), sampling=SamplingConfig(bias_to_high_gradients=1.3, sample_points_per_snapshot=150), output=OutputConfig(s3_endpoint='https://s3-west.nrp-nautilus.io', bucket='dbof/', folder='native_grid_dbof_training_data/', dataset_name='cutout_dataset_creation.zarr', target_km_res=150, down_sample_res=64), features=FeaturesConfig(feature_channels=['Eta', 'Salt', 'Theta', 'U', 'V', 'W', 'gradb2']), runtime=RuntimeConfig(zarr_async_concurrency=128, dask_scheduler='distributed', dask_n_workers=None, dask_threads_per_worker=None, dask_memory_limit=None))
2026-06-25 08:21:48,306 | INFO | Log file: C:\Users\Jake T


  0%|          | 0/1 [16:34<?, ?it/s]
2026-06-25 08:42:08,040 - distributed.worker - ERROR - Failed to communicate with scheduler during heartbeat.
Traceback (most recent call last):
  File "C:\Users\Jake Tallman\PycharmProjects\dbof-in-native-grid\.venv1\Lib\site-packages\distributed\comm\tcp.py", line 226, in read
    frames_nosplit_nbytes_bin = await stream.read_bytes(fmt_size)
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
tornado.iostream.StreamClosedError: Stream is closed

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\Jake Tallman\PycharmProjects\dbof-in-native-grid\.venv1\Lib\site-packages\distributed\worker.py", line 1273, in heartbeat
    response = await retry_operation(
               ^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Jake Tallman\PycharmProjects\dbof-in-native-grid\.venv1\Lib\site-packages\distributed\utils_comm.py", line 416, in retry_operation
    return await retry(
   

2## Example run 2 - very small
- 2 weeks of data
- 1 snapshot per week
- 10 cutouts per timestamp

If you are running this on your laptop or pc expect it to take a while.

In [6]:
run_id = f"small_test{datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S')}"
print(f"Your run id is: {run_id}")
print("Make sure and use the same run_d later when accessing your data")

Your run id is: small_test20260203_160339
Make sure and use the same run_d later when accessing your data


In [ ]:
!generate-llc-dataset --config ../../configs/cutouts/run/test.yaml --run_id {run_id}

In [ ]:
# aws cli arguments for listing or deleting data in the s3 bucket

# aws --endpoint https://s3-west.nrp-nautilus.io s3 ls s3://llc/native_grid_dbof_training_data/script_test_00/ --human-readable
#
# aws --endpoint https://s3-west.nrp-nautilus.io s3 rm s3://llc/native_grid_dbof_training_data/script_test_00/ --recursive --dryrun